In [0]:
%run ./connectionNotebook

In [0]:
catalog_name = 'adbrag'
target_schema_name = 'gold'
src_schema_name = 'silver'

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, upper, year, current_date

silver_df = spark.table(f"{catalog_name}.{src_schema_name}.employees_silver")

In [0]:
gold_df = (
    silver_df
    .withColumn(
        "salary_band",
        when(col("src_Salary") < 40000, "Low")
        .when((col("src_Salary") >= 40000) & (col("src_Salary") < 80000), "Medium")
        .otherwise("High")
    )
    .withColumn("department_standardized", upper(col("src_Department")))
    .withColumn("years_of_service", year(current_date()) - year(col("src_HireDate")))
    .withColumn("gold_loaded_ts", current_timestamp())
)



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{target_schema_name}.employees_gold
(
    src_EmployeeID INT,
    src_EmployeeName STRING,
    src_Department STRING,
    src_HireDate DATE,
    src_Salary DECIMAL(10,2),
    processed_ts TIMESTAMP,
    salary_band STRING,
    department_standardized STRING,
    years_of_service INT,
    gold_loaded_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_EmployeeID)
""")



In [0]:
gold_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{target_schema_name}.employees_gold")